In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Setup & Imports

Patches

In [3]:
import sys

# TODO: Temporary patch to get rf2aa, biotite and cifutils working from the rf2aa .sif container
sys.path.append("/home/smathis/code/RF2-allatom/rf2aa")  # for rf2aa
sys.path.append("/mnt/home/smathis/code/biotite/src/")  # for biotite
sys.path.append("/home/smathis/miniconda3/lib/python3.12/site-packages/")  # for cifutils

Actual imports

In [5]:
from datahub.encoding_definitions import *  # noqa
from datahub.transforms.base import *  # noqa
from datahub.utils.token import *  # noqa
from datahub.transforms._viz_utils import *  # noqa
from datahub.transforms.encoding import *  # noqa
from datahub.transforms.atom_array import *  # noqa
from datahub.transforms.atomize import *  # noqa
from datahub.transforms.bonds import *  # noqa
from datahub.transforms.crop import *  # noqa
from datahub.transforms.covalent_modifications import *  # noqa
from datahub.transforms.symmetry import *  # noqa
from datahub.transforms.template import *  # noqa
from datahub.transforms.openbabel_utils import *  # noqa
from datahub.transforms.chirals import *  # noqa
import numpy as np
import matplotlib.pyplot as plt
from datahub.datasets import *
from datahub.datasets.dataframe_parsers import *
from functools import cache
from copy import deepcopy
from datahub.utils.rng import create_rng_state_from_seeds, rng_state
from pathlib import Path
from typing import *

# Turn off all logging
import logging

logger = logging.getLogger()
logger.setLevel(logging.CRITICAL)

# Loading CIF files

In [6]:
from cifutils.cifutils_biotite.cifutils_biotite import CIFParser

# NOTE: This takes a few seconds to load (~5-10s) -- we'll speed this up in the future
CIF_PARSER = CIFParser()


def get_digs_path(pdbid: str, base: Literal["trrosetta", "mirror"] = "trrosetta") -> str:
    if base == "trrosetta":
        base_dir = Path("/databases/TrRosetta/cif")
    elif base == "mirror":
        base_dir = Path("/databases/rcsb/cif")
    else:
        raise ValueError(f"Invalid base: {base}")
    pdbid = pdbid.lower()
    filename = f"{base_dir}/{pdbid[1:3]}/{pdbid}.cif.gz"
    if not os.path.exists(filename):
        raise ValueError(f"File {filename} does not exist")
    return filename


def parse(
    pdb_id: str,
    base: Literal["trrosetta", "mirror"] = "trrosetta",
    convert_mse_to_met: bool = True,
    remove_waters: bool = True,
    remove_crystallization_aids: bool = True,
    build_assembly: str = "first",
) -> AtomArray:
    data = CIF_PARSER.parse(
        get_digs_path(pdb_id, base),
        convert_mse_to_met=convert_mse_to_met,
        remove_waters=remove_waters,
        remove_crystallization_aids=remove_crystallization_aids,
        build_assembly=build_assembly,
    )
    # Grab bioassembly and populate it into atom_array slot
    if "atom_array" not in data:
        assembly_ids = list(data["assemblies"].keys())
        data["atom_array"] = data["assemblies"][assembly_ids[0]][0]
    data["pdb_id"] = pdb_id
    return data

In [7]:
data = parse("6lyz")

In [8]:
view(data["atom_array"])

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

# Loading PDB files

In [9]:
from biotite.structure.io import pdb
from biotite.database import rcsb
from io import StringIO

In [10]:
# Fetch PDB file from RCSB
pdb_str = rcsb.fetch("6lyz", "pdb")  #  this is a stirng IO object, but could be a local file instead

# Load PDB file
pdb_file = pdb.PDBFile.read(pdb_str)
structure = pdb.get_structure(
    pdb_file,
    model=1,
    altloc="occupancy",
    extra_fields=["occupancy", "atom_id", "b_factor", "charge"],
    include_bonds=True,
)
structure

AtomArray([
	Atom(np.array([ 3.287, 10.092, 10.329], dtype=float32), chain_id="A", res_id=1, ins_code="", res_name="LYS", hetero=False, atom_name="N", element="N", occupancy=1.0, atom_id=1, b_factor=5.89, charge=0),
	Atom(np.array([ 2.445, 10.457,  9.182], dtype=float32), chain_id="A", res_id=1, ins_code="", res_name="LYS", hetero=False, atom_name="CA", element="C", occupancy=1.0, atom_id=2, b_factor=8.16, charge=0),
	Atom(np.array([ 2.5  , 11.978,  9.038], dtype=float32), chain_id="A", res_id=1, ins_code="", res_name="LYS", hetero=False, atom_name="C", element="C", occupancy=1.0, atom_id=3, b_factor=8.04, charge=0),
	Atom(np.array([ 2.588, 12.719, 10.041], dtype=float32), chain_id="A", res_id=1, ins_code="", res_name="LYS", hetero=False, atom_name="O", element="O", occupancy=1.0, atom_id=4, b_factor=7.07, charge=0),
	Atom(np.array([1.006, 9.995, 9.385], dtype=float32), chain_id="A", res_id=1, ins_code="", res_name="LYS", hetero=False, atom_name="CB", element="C", occupancy=1.0, atom_i

In [9]:
# Check bonds:
structure.bonds.as_array()  # <-- bonds exist

array([[  2,   9,   1],
       [ 11,  16,   1],
       [ 18,  27,   1],
       ...,
       [237, 888,   0],
       [512, 629,   0],
       [600, 723,   0]], dtype=uint32)

In [10]:
view(structure)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

# Example transforms

In [11]:
seed = 3
data = parse("6lyz")
encoding = RF2_ATOM36_ENCODING
pipe = Compose(
    [
        RemoveHydrogens(),
        AddGlobalAtomIdAnnotation(),
        FilterAndAnnotatePNUnits(),
        RemoveTerminalOxygen(),
        AddChainTypeAnnotation(),
        RegenerateChainEntities(),
        AddChainEntityIdAnnotation(),
        AddMoleculeAnnotation(),
        AddMoleculeEntityAnnotation(),
        AtomizeResidues(atomize_by_default=True, res_names_to_ignore=encoding.tokens, move_atomized_part_to_end=True),
        AddGlobalTokenIdAnnotation(),
        SortPolyThenNonPoly(),
        AddOpenBabelMoleculesForAtomizedMolecules(),
        CropSpatialLikeAF3(crop_size=50, keep_uncropped_atom_array=True),
        EncodeAtomArray(encoding=encoding),
    ],
    track_rng_state=False,
)

with rng_state(create_rng_state_from_seeds(np_seed=seed, torch_seed=seed, py_seed=seed)):
    data = pipe(data)

In [12]:
# Look at cropped output
view(data["atom_array"])

3Dmol.js failed to load for some reason. Please check your browser console for error messages.